# Partie A — Cavité entraînée à $Re = 100$ : solveur de référence vérifié

Plan du notebook (cf. `PLAN.md`, étapes 1 à 3) :

1. **Vérification du solveur de Poisson** (gradient conjugué sans matrice, `cavity/common.py`)
   sur des solutions connues, en Dirichlet (pour $\psi$) et en Neumann (pour $p$).
2. M1 (projection) à $Re = 100$ pour $N = 32, 64, 128$ : profils vs Ghia, justification de la grille,
   $K(t)$, $P(t)$, $P$ stationnaire en fonction de $\ln N$.
3. M2 ($\psi$–$\omega$, Thom) et comparaison M1 / M2 / Ghia.

Adimensionnement : $L = U = \rho = 1$, $\nu = 1/Re$. Grille colocalisée aux nœuds, $h = 1/N$.

In [ ]:
# Imports et chemins : le notebook est dans notebooks/, le paquet cavity/ est à la racine du projet
import os
# Contournement d'un défaut de l'environnement CFD sur cette machine : la couche de threads OpenMP
# de MKL fait planter toute algèbre linéaire numpy (np.linalg, donc aussi matplotlib) ; on force
# MKL en séquentiel. Doit être fixé AVANT le premier import de numpy. Sans effet sur nos calculs.
os.environ["MKL_THREADING_LAYER"] = "SEQUENTIAL"
import sys
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
FIG = ROOT / "figures"
RES = ROOT / "results"
FIG.mkdir(exist_ok=True)
RES.mkdir(exist_ok=True)

from cavity.common import solve_poisson

GRIDS_POISSON = (16, 32, 64)       # grilles demandées pour la vérification du Poisson
TOL_CG = 1e-10                     # résidu relatif : bien sous l'erreur de discrétisation attendue


def observed_slopes(n_list, errors):
    # Pente observée entre deux grilles successives (h divisé par 2) : log2(e_N / e_2N),
    # et pente globale par régression linéaire de log(e) en fonction de log(h) (workshop 9).
    # Régression écrite à la main : pente des moindres carrés = cov(log h, log e) / var(log h).
    e = np.asarray(errors, dtype=float)
    local = np.log2(e[:-1] / e[1:])
    log_h = np.log(1.0 / np.asarray(n_list, dtype=float))
    log_e = np.log(e)
    dh = log_h - log_h.mean()
    global_fit = np.sum(dh * (log_e - log_e.mean())) / np.sum(dh * dh)
    return local, global_fit

## 1. Vérification du solveur de Poisson

### 1.a Dirichlet : $\phi = (x^2-1)(y^2-1)$ (workshop 11.1)

**Ce qu'on fait.** On résout $\Delta\phi = f = 2(x^2 + y^2 - 2)$ sur $[-1, 1]^2$ avec $\phi = 0$ au bord
(problème du workshop 11.1, donc sur $[-1,1]^2$ et non sur le carré unité : c'est là que $\phi$ s'annule au bord ;
le pas vaut $h = 2/N$). Départ du gradient conjugué de zéro, tolérance $10^{-10}$ sur le résidu relatif.

**Ce qu'on attend.** Attention : l'erreur de troncature du laplacien 5 points est
$\tfrac{h^2}{12}(\partial^4_x\phi + \partial^4_y\phi) + O(h^4)$ (cours §2.3), et ici $\phi$ est de degré 2 en
chaque variable : **le schéma est exact**. L'erreur ne mesure donc que l'arrêt du gradient conjugué
($\sim 10^{-12}$) ; il n'y a **pas de pente 2** à observer sur ce cas. Il sert à vérifier que le solveur
(stencil, conditions aux limites, gradient conjugué) est juste. Nombre d'itérations attendu : $O(N)$
(complexité $O(n^{3/2})$ avec $n = N^2$ inconnues, cours §4.8).

In [ ]:
rows_dir = []
for n in GRIDS_POISSON:
    h = 2.0 / n
    x = np.linspace(-1.0, 1.0, n + 1)
    X, Y = np.meshgrid(x, x, indexing="ij")          # [i, j] <-> (x, y)
    phi_exact = (X**2 - 1.0) * (Y**2 - 1.0)
    f = 2.0 * (X**2 + Y**2 - 2.0)
    phi, n_iter = solve_poisson(f[1:-1, 1:-1], h, "dirichlet", tol=TOL_CG)
    err = np.abs(phi - phi_exact).max()
    rows_dir.append((n, err, n_iter))
    print(f"N = {n:3d}   max|phi - phi_exact| = {err:.3e}   itérations GC = {n_iter}")

**Lecture.** Erreur $\sim 10^{-12}$ sur les trois grilles, indépendante de $N$ : c'est la précision d'arrêt du
gradient conjugué, comme prévu pour un schéma exact sur ce polynôme. Le nombre d'itérations double
quand $N$ double : $O(N)$, conforme au cours (§4.8).

### 1.b Complément Dirichlet : $\phi = \sin(\pi x)\sin(\pi y)$ sur $[0, 1]^2$

**Ce qu'on fait.** Le cas 1.a ne peut pas donner d'ordre. On ajoute donc un cas dont la dérivée quatrième
n'est pas nulle : $\Delta\phi = -2\pi^2\phi$, $\phi = 0$ sur les parois du carré unité (notre domaine).

**Ce qu'on attend.** Pente $\approx 2$. Remarque : $\sin(\pi x_i)\sin(\pi y_j)$ est un **vecteur propre** du laplacien
discret en Dirichlet, donc le gradient conjugué converge en **1 itération** (le résidu initial est colinéaire à
un vecteur propre) : ce cas ne sert pas à compter les itérations, c'est le cas 1.a qui le fait.

In [ ]:
rows_sin = []
for n in GRIDS_POISSON:
    h = 1.0 / n
    x = np.linspace(0.0, 1.0, n + 1)
    X, Y = np.meshgrid(x, x, indexing="ij")
    phi_exact = np.sin(np.pi * X) * np.sin(np.pi * Y)
    phi, n_iter = solve_poisson((-2.0 * np.pi**2 * phi_exact)[1:-1, 1:-1], h, "dirichlet", tol=TOL_CG)
    err = np.abs(phi - phi_exact).max()
    rows_sin.append((n, err, n_iter))
    print(f"N = {n:3d}   max|phi - phi_exact| = {err:.3e}   itérations GC = {n_iter}")

local, glob = observed_slopes(GRIDS_POISSON, [r[1] for r in rows_sin])
print("pentes locales :", np.round(local, 3), "  pente globale :", round(glob, 3))

### 1.c Neumann : $\phi = \cos(\pi x)\cos(\pi y)$ sur $[0, 1]^2$

**Ce qu'on fait.** $\Delta\phi = -2\pi^2\phi$ avec $\partial\phi/\partial n = 0$ sur les quatre parois (vrai pour
cette $\phi$), exactement comme pour la pression de M1 : bord = voisin intérieur (Neumann décentrée
d'ordre 1, cours §4.2.2.2), moyenne du second membre retirée, $\phi$ à moyenne nulle à chaque itération.
La moyenne de $\phi_\text{exact}$ sur les nœuds intérieurs est nulle (symétrie), on peut donc comparer directement.
Erreur mesurée sur les nœuds intérieurs (les nœuds de bord sont une simple recopie).

**Ce qu'on attend.** On pourrait attendre une pente $\approx 2$ (ordre du laplacien), mais l'analyse avant calcul prévoit plutôt
**une pente $\approx 1$** : la règle $\phi_0 = \phi_1$ impose $\partial\phi/\partial n = 0$ **entre** les nœuds 0 et 1,
c'est-à-dire que le schéma résout (à l'ordre 2) le problème de Neumann sur le domaine décalé
$[h/2,\ 1-h/2]^2$. En 1D, la solution de $\phi'' = -\pi^2\cos(\pi x)$ avec $\phi'(h/2) = \phi'(1-h/2) = 0$ vaut
$\cos(\pi x) + \pi\sin(\pi h/2)\,x + c$ : l'écart à la vraie solution est une **pente linéaire en $x$**,
d'amplitude $\approx \tfrac{\pi^2}{2}h\,(x - \tfrac12)$, donc une erreur maximale $\approx \tfrac{\pi^2}{4}h \approx 2.5\,h$ en 1D.

In [ ]:
rows_neu = []
profiles_neu = {}                     # erreur le long de y = 0.5 (pour la figure)
for n in GRIDS_POISSON:
    h = 1.0 / n
    x = np.linspace(0.0, 1.0, n + 1)
    X, Y = np.meshgrid(x, x, indexing="ij")
    phi_exact = np.cos(np.pi * X) * np.cos(np.pi * Y)
    phi, n_iter = solve_poisson((-2.0 * np.pi**2 * phi_exact)[1:-1, 1:-1], h, "neumann", tol=TOL_CG)
    error = phi - phi_exact
    err = np.abs(error[1:-1, 1:-1]).max()
    rows_neu.append((n, err, n_iter))
    profiles_neu[n] = (x, error[:, n // 4])          # coupe y = 0.25 (cos(pi y) = 0 en y = 0.5)
    print(f"N = {n:3d}   max|phi - phi_exact| (intérieur) = {err:.3e}   err/h = {err / h:.2f}"
          f"   itérations GC = {n_iter}   moyenne(phi) = {phi[1:-1, 1:-1].mean():+.1e}")

local, glob = observed_slopes(GRIDS_POISSON, [r[1] for r in rows_neu])
print("pentes locales :", np.round(local, 3), "  pente globale :", round(glob, 3))

**Figure** : erreur max en fonction de $h$ (log-log) pour Dirichlet ($\sin\sin$) et Neumann ($\cos\cos$), avec
les droites de pente 1 et 2 ; à droite, l'erreur Neumann le long de la coupe $y = 1/4$ divisée par $h$
(si l'analyse du domaine décalé est juste, les courbes se superposent : erreur $\propto h$, de forme régulière).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

ax = axes[0]
h_list = 1.0 / np.array(GRIDS_POISSON, dtype=float)
e_sin = np.array([r[1] for r in rows_sin])
e_neu = np.array([r[1] for r in rows_neu])
ax.loglog(h_list, e_sin, "o-", label=r"Dirichlet, $\sin(\pi x)\sin(\pi y)$")
ax.loglog(h_list, e_neu, "s-", label=r"Neumann, $\cos(\pi x)\cos(\pi y)$")
ax.loglog(h_list, e_sin[0] * (h_list / h_list[0])**2, "k--", lw=0.8, label="pente 2")
ax.loglog(h_list, e_neu[0] * (h_list / h_list[0])**1, "k:", lw=0.8, label="pente 1")
ax.set_xlabel("$h$")
ax.set_ylabel(r"$\max|\phi - \phi_\mathrm{exact}|$")
ax.set_title("Convergence en maillage du Poisson discret")
ax.legend(fontsize=8)
ax.grid(True, which="both", alpha=0.3)

ax = axes[1]
for n, (x, err_line) in profiles_neu.items():
    ax.plot(x, err_line * n, label=f"N = {n}")     # erreur / h
ax.set_xlabel("$x$")
ax.set_ylabel(r"$(\phi - \phi_\mathrm{exact}) / h$ en $y = 1/4$")
ax.set_title("Neumann : forme de l'erreur")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(FIG / "A_poisson_convergence.png", dpi=150)
plt.show()

# Résultats numériques
table = np.array([[n, ed, it_d, es, it_s, en, it_n]
                  for (n, ed, it_d), (_, es, it_s), (_, en, it_n) in zip(rows_dir, rows_sin, rows_neu)])
np.savetxt(RES / "A_poisson_verification.csv", table, delimiter=",", fmt="%.6e",
           header="N,err_dirichlet_quadratique,iter_dirichlet_quadratique,err_dirichlet_sinsin,"
                  "iter_dirichlet_sinsin,err_neumann_coscos,iter_neumann_coscos")

**Interprétation.**

- **Dirichlet** : pente 2 sur $\sin\sin$, l'ordre attendu du laplacien 5 points ; avec le cas exact 1.a, le
  solveur Dirichlet (utilisé pour $\psi$ dans M2) est vérifié.
- **Neumann** : pente $\approx 1$ (0,82 puis 0,92, globale 0,87 : elle tend vers 1 quand $h$ diminue), et non 2.
  Ce n'est pas un défaut du gradient conjugué : la règle $\phi_0 = \phi_1$ (Neumann décentrée d'ordre 1,
  cours §4.2.2.2) revient à placer la paroi en $h/2$. Les courbes « erreur / $h$ » des trois grilles se
  superposent, ce qui confirme une erreur **globale** en $O(h)$, régulière et étalée sur tout le domaine
  (pas seulement la rangée de bord). Erreur max mesurée $\approx 2.3\,h$ à $2.7\,h$ ; l'estimation 1D donne
  $\approx 2.5\,h$, du même ordre (en 2D la correction n'est pas exactement la pente linéaire du cas 1D,
  on ne prétend donc qu'à l'ordre de grandeur).
- **Conséquence pour M1** : la pression de M1 sera au mieux d'ordre 1. Cela ne dit pas encore que les
  **vitesses** le seront : c'est ce que mesurera l'étude de grille de la Partie A (étape 2), puis la comparaison
  avec M2 (qui n'a pas de Neumann). Conformément au PLAN, on ne change rien tant que cette mesure n'est pas faite.
- **Itérations** : $O(N)$ pour les deux conditions aux limites (départ de zéro). En marche en temps, le départ
  à chaud depuis la pression du pas précédent devrait en réduire fortement le nombre.

## 2. M1 (projection) à $Re = 100$ : validation et justification de la grille

### 2.a Calculs pour $N = 32, 64, 128$

**Ce qu'on fait.** M1 (`cavity/projection.py`, cours §7.3.2 et workshop 15) : Euler explicite, advection
**centrée**, diffusion centrée, pression par gradient conjugué Neumann (tol $= 10^{-8}$, départ à chaud).
Départ du repos, couvercle $U_t = 1 - e^{-t}$. Arrêt quand $\max|u^{n+1}-u^n| < 10^{-6}$ et $t > 10$
(`docs/adr/0003-pas-de-temps-et-arrets.md`). $\Delta t$ = limites de stabilité du cours (§5.3), marge 0.8.

**Ce qu'on attend.**
- $\Delta t$ limité par l'advection (CFL) à $N = 32$ et par la diffusion (Fourier) à $N = 64, 128$ (croisement à $N = Re/2 = 50$).
- Coût : nombre de pas $\propto 1/\Delta t \propto N^2$, coût d'un pas $\propto N^2 \times$ (itérations du gradient conjugué $\propto N$),
  donc environ $N^5$ au-delà de $N = 50$ : $N = 128$ devrait coûter de l'ordre de 30 fois $N = 64$.

Les résultats sont sauvegardés dans `results/` et relus si présents (`RECOMPUTE = True` pour tout recalculer).

In [ ]:
from cavity.projection import run_projection

RE = 100
NU = 1.0 / RE
GRIDS = (32, 64, 128)
CG_TOL = 1e-8
RECOMPUTE = False


def save_run(run, path):
    np.savez(path, **{key: np.asarray(val) for key, val in run.items()})


def load_run(path):
    data = np.load(path)
    return {key: (data[key].item() if data[key].ndim == 0 else data[key]) for key in data.files}


def get_run(n, cg_tol=CG_TOL):
    # Relit le calcul s'il est déjà dans results/, sinon le lance et le sauvegarde.
    path = RES / f"A_M1_N{n}_tol{cg_tol:.0e}.npz"
    if path.exists() and not RECOMPUTE:
        return load_run(path)
    run = run_projection(n, RE, "centered", cg_tol=cg_tol)
    save_run(run, path)
    return run


def active_limit(h, nu):
    # Quelle limite de stabilité fixe Δt (cf. cavity/common.time_step)
    limits = {"Fourier": h**2 / (4 * nu), "CFL": h / 2, "centré 2ν": 2 * nu}
    return min(limits, key=limits.get)


runs = {n: get_run(n) for n in GRIDS}

print(f"{'N':>4} {'dt':>9} {'limite':>9} {'pas':>7} {'t_final':>8} {'calcul (s)':>10} "
      f"{'it. GC moy':>10} {'max|du/dt| fin':>14} {'convergé':>8}")
for n, r in runs.items():
    print(f"{n:4d} {r['dt']:9.3e} {active_limit(r['h'], NU):>9} {r['n_steps']:7d} {r['t_final']:8.2f} "
          f"{r['wall_time']:10.1f} {r['cg_iterations'].mean():10.1f} {r['max_dudt']:14.2e} {str(r['converged']):>8}")

**Interprétation.** *(à rédiger après lecture des résultats)*

### 2.b Profils sur les lignes médianes, comparés à Ghia et al. (1982)

**Ce qu'on fait.** $u(0.5, y)$ et $v(x, 0.5)$ pour les trois grilles ($x = 0.5$ et $y = 0.5$ sont des nœuds pour $N$ pair),
avec les points de Ghia (`ref_data/`, Tables I et II, colonne $Re = 100$, grille 129×129).
Les abscisses de Ghia sont des multiples de $1/128$ arrondis à 4 décimales : on les remet à leur valeur exacte
$\mathrm{round}(128\,y)/128$. Pour comparer, notre profil est interpolé linéairement aux points de Ghia
(exact pour $N = 128$ ; pour $N = 32, 64$, interpolation d'ordre 2, du même ordre que le schéma).

**Ce qu'on attend.** Les courbes se rapprochent de Ghia quand $N$ augmente ; à $Re = 100$, le tourbillon est
presque centré, $u$ minimal $\approx -0.21$ vers $y \approx 0.45$ (valeur lue dans la table de Ghia, pas inventée).

In [ ]:
ghia_u = np.loadtxt(ROOT / "ref_data" / "ghia1982_u_vertical_centerline.csv", delimiter=",")
ghia_v = np.loadtxt(ROOT / "ref_data" / "ghia1982_v_horizontal_centerline.csv", delimiter=",")
y_ghia = np.round(ghia_u[:, 0] * 128) / 128          # positions exactes sur la grille 129 de Ghia
x_ghia = np.round(ghia_v[:, 0] * 128) / 128
u_ghia = ghia_u[:, 1]                                 # colonne Re = 100
v_ghia = ghia_v[:, 1]


def centerlines(run):
    # u(0.5, y) : colonne i = N/2 ; v(x, 0.5) : ligne j = N/2
    n = run["N"]
    return run["u"][n // 2, :], run["v"][:, n // 2]


def ghia_errors(run):
    # Écart max à Ghia sur ses points intérieurs (les points de paroi sont imposés, donc exacts)
    u_c, v_c = centerlines(run)
    in_u = (y_ghia > 0) & (y_ghia < 1)
    in_v = (x_ghia > 0) & (x_ghia < 1)
    err_u = np.abs(np.interp(y_ghia[in_u], run["y"], u_c) - u_ghia[in_u]).max()
    err_v = np.abs(np.interp(x_ghia[in_v], run["x"], v_c) - v_ghia[in_v]).max()
    return err_u, err_v


fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for n, r in runs.items():
    u_c, v_c = centerlines(r)
    axes[0].plot(u_c, r["y"], label=f"M1, N = {n}")
    axes[1].plot(r["x"], v_c, label=f"M1, N = {n}")
axes[0].plot(u_ghia, y_ghia, "ko", mfc="none", label="Ghia et al. (1982)")
axes[1].plot(x_ghia, v_ghia, "ko", mfc="none", label="Ghia et al. (1982)")
axes[0].set_xlabel("$u(0.5, y)$"); axes[0].set_ylabel("$y$")
axes[1].set_xlabel("$x$"); axes[1].set_ylabel("$v(x, 0.5)$")
axes[0].set_title("Ligne verticale médiane"); axes[1].set_title("Ligne horizontale médiane")
for ax in axes:
    ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
fig.suptitle(f"M1, Re = {RE} : profils stationnaires")
fig.tight_layout()
fig.savefig(FIG / "A_M1_profiles.png", dpi=150)
plt.show()

for n, r in runs.items():
    eu, ev = ghia_errors(r)
    print(f"N = {n:3d}   max|u - u_Ghia| = {eu:.4f}   max|v - v_Ghia| = {ev:.4f}")

**Interprétation.** *(à rédiger après lecture des résultats)*

### 2.c Justification de la grille : deux courbes de convergence

**Ce qu'on fait** (PLAN.md, étape 2) :
1. écart max à Ghia en fonction de $h$ (log-log) ;
2. écart entre grilles successives $\max|u_N - u_{2N}|$ (et de même pour $v$), sur les nœuds communs des lignes
   médianes : il ne dépend d'aucune référence, sa pente donne **notre** ordre observé.

**Ce qu'on attend.** Schéma d'ordre 2 en espace ⇒ pente $\approx 2$. Trois raisons possibles d'une pente plus faible :
la Neumann d'ordre 1 sur $p$ (cf. ADR 0002, à trancher avec M2 à l'étape 3), les singularités des coins du
couvercle (touchent M1 et M2), et pour la courbe 1, l'erreur propre de Ghia (palier possible).
Avec 3 grilles, la courbe 1 donne 2 pentes locales et la courbe 2 une seule.

In [ ]:
h_list = np.array([runs[n]["h"] for n in GRIDS])
err_ghia = np.array([ghia_errors(runs[n]) for n in GRIDS])        # colonnes : u, v

# Écart entre grilles successives sur les nœuds communs (un nœud sur deux de la grille fine)
diff_succ = []
for n in GRIDS[:-1]:
    u_c, v_c = centerlines(runs[n])
    u_f, v_f = centerlines(runs[2 * n])
    diff_succ.append((np.abs(u_c - u_f[::2]).max(), np.abs(v_c - v_f[::2]).max()))
diff_succ = np.array(diff_succ)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
for k, name in enumerate(("u", "v")):
    ax = axes[k]
    ax.loglog(h_list, err_ghia[:, k], "o-", label=rf"$\max|{name} - {name}_\mathrm{{Ghia}}|$")
    ax.loglog(h_list[:-1], diff_succ[:, k], "s-", label=rf"$\max|{name}_N - {name}_{{2N}}|$ (en $h_N$)")
    ref = err_ghia[0, k]
    ax.loglog(h_list, ref * (h_list / h_list[0])**2, "k--", lw=0.8, label="pente 2")
    ax.loglog(h_list, ref * (h_list / h_list[0])**1, "k:", lw=0.8, label="pente 1")
    ax.set_xlabel("$h$"); ax.set_title(f"Convergence en maillage de ${name}$ (M1)")
    ax.grid(True, which="both", alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "A_M1_grid_convergence.png", dpi=150)
plt.show()

for k, name in enumerate(("u", "v")):
    local, glob = observed_slopes(GRIDS, err_ghia[:, k])
    succ_slope = np.log2(diff_succ[0, k] / diff_succ[1, k])
    print(f"{name} : écart à Ghia = {np.round(err_ghia[:, k], 5)}  pentes locales {np.round(local, 2)}"
          f"  (globale {glob:.2f})")
    print(f"    écart grilles successives = {np.round(diff_succ[:, k], 5)}  pente {succ_slope:.2f}")

**Interprétation.** *(à rédiger après lecture des résultats)*

### 2.d $K(t)$ et $P(t)$, et approche du stationnaire

**Ce qu'on fait.** Les deux diagnostics de l'énoncé, à chaque pas de temps, pour les trois grilles. À droite :
$\max|\partial_t u| = \max|u^{n+1}-u^n|/\Delta t$ en échelle log, avec le seuil d'arrêt $10^{-6}/\Delta t$ de chaque grille
(cf. ADR 0003 : le critère dépend de $\Delta t$).

**Ce qu'on attend.** $K$ et $P$ montent avec la rampe du couvercle (temps caractéristique 1), puis tendent vers un
palier. $K$ devrait peu dépendre de $N$ (grandeur intégrale, régulière) ; $P$ devrait **croître** avec $N$
(singularité des coins). $\max|\partial_t u|$ devrait décroître exponentiellement à la fin.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for n, r in runs.items():
    axes[0].plot(r["t"], r["K"], label=f"N = {n}")
    axes[1].plot(r["t"], r["P"], label=f"N = {n}")
    line, = axes[2].semilogy(r["t"], r["change"] / r["dt"], label=f"N = {n}")
    axes[2].axhline(1e-6 / r["dt"], color=line.get_color(), ls=":", lw=0.8)
axes[0].set_ylabel("$K(t)$"); axes[1].set_ylabel("$P(t)$")
axes[2].set_ylabel(r"$\max|\partial_t u|$  (pointillés : seuil d'arrêt)")
for ax in axes:
    ax.set_xlabel("$t$"); ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
axes[0].set_title("Énergie cinétique"); axes[1].set_title("Puissance injectée par le couvercle")
axes[2].set_title("Approche du stationnaire")
fig.tight_layout()
fig.savefig(FIG / "A_M1_K_P.png", dpi=150)
plt.show()

# Ce qu'il reste à parcourir après l'arrêt : si max|∂t u| ~ A e^{-λt}, l'écart restant au stationnaire
# vaut ∫_t^∞ A e^{-λs} ds = max|∂t u|(t_final) / λ. λ : régression de ln(max|∂t u|) sur le dernier quart.
for n, r in runs.items():
    rate = r["change"] / r["dt"]
    last = slice(3 * len(rate) // 4, None)
    tt, lr = r["t"][last], np.log(rate[last])
    lam = -np.sum((tt - tt.mean()) * (lr - lr.mean())) / np.sum((tt - tt.mean())**2)
    print(f"N = {n:3d}   K_final = {r['K'][-1]:.6f}   P_final = {r['P'][-1]:.6f}   "
          f"λ = {lam:.3f}   écart restant estimé ≈ {rate[-1] / lam:.1e}")

**Interprétation.** *(à rédiger après lecture des résultats)*

### 2.e $P$ stationnaire en fonction de $\ln N$

**Ce qu'on fait.** On trace $P_\text{final}$ contre $\ln N$ et on ajuste $P = a + c\ln N$ (régression sur 3 points).

**Ce qu'on attend** (PLAN.md §2, « Singularités aux coins du couvercle ») : aux deux coins, la vitesse de paroi saute
de $U_t$ à 0, $\partial_y u \sim 1/r$, donc $\int \nu\,\partial_y u\,dx$ diverge comme $\ln$ en continu. La grille coupe à $r \approx h$ :
$P \approx P_\text{régulier} + c\,\nu\ln N$. On attend donc **une droite, sans palier** : $P$ ne converge pas en maillage
et ne peut pas servir à choisir $N$. Le choix de $N$ se fait sur les profils (2.c) et sur $K$.

In [ ]:
n_arr = np.array(GRIDS, dtype=float)
p_final = np.array([runs[n]["P"][-1] for n in GRIDS])
k_final = np.array([runs[n]["K"][-1] for n in GRIDS])
ln_n = np.log(n_arr)
c_fit = np.sum((ln_n - ln_n.mean()) * (p_final - p_final.mean())) / np.sum((ln_n - ln_n.mean())**2)
a_fit = p_final.mean() - c_fit * ln_n.mean()

fig, ax = plt.subplots(figsize=(5.5, 4))
ax.plot(ln_n, p_final, "o", label="M1")
ax.plot(ln_n, a_fit + c_fit * ln_n, "k--", lw=0.8, label=rf"$P = {a_fit:.4f} + {c_fit:.4f}\,\ln N$")
for n, lp, pp in zip(GRIDS, ln_n, p_final):
    ax.annotate(f"N = {n}", (lp, pp), textcoords="offset points", xytext=(5, -12), fontsize=8)
ax.set_xlabel(r"$\ln N$"); ax.set_ylabel(r"$P$ stationnaire")
ax.set_title(f"M1, Re = {RE} : P en fonction de ln N")
ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "A_M1_P_lnN.png", dpi=150)
plt.show()

print("P_final :", np.round(p_final, 6), "  accroissements :", np.round(np.diff(p_final), 6))
print(f"pente c = {c_fit:.5f}   c / ν = {c_fit / NU:.3f}")
print("K_final :", np.round(k_final, 6), "  accroissements :", np.round(np.diff(k_final), 6))

**Interprétation.** *(à rédiger après lecture des résultats)*

### 2.f Effet de la tolérance du gradient conjugué (workshop 15)

**Ce qu'on fait** (ADR 0003) : on refait $N = 128$, la grille la plus exigeante, avec tol $= 10^{-10}$ au lieu de $10^{-8}$,
et on compare les profils.

**Ce qu'on attend.** Si l'écart entre les deux calculs est très petit devant l'écart à Ghia, une tolérance fixe de
$10^{-8}$ suffit pour toutes nos grilles. Le calcul coûte plus cher (plus d'itérations par pas).

In [ ]:
r8 = runs[128]
r10 = get_run(128, cg_tol=1e-10)
u8, v8 = centerlines(r8)
u10, v10 = centerlines(r10)
du_tol = np.abs(u8 - u10).max()
dv_tol = np.abs(v8 - v10).max()
eu8, ev8 = ghia_errors(r8)
print(f"tol 1e-8  : it. GC moy = {r8['cg_iterations'].mean():6.1f}   calcul = {r8['wall_time']:7.1f} s   "
      f"t_final = {r8['t_final']:.2f}")
print(f"tol 1e-10 : it. GC moy = {r10['cg_iterations'].mean():6.1f}   calcul = {r10['wall_time']:7.1f} s   "
      f"t_final = {r10['t_final']:.2f}")
print(f"max|u(1e-8) - u(1e-10)| = {du_tol:.2e}   (écart à Ghia : {eu8:.2e})")
print(f"max|v(1e-8) - v(1e-10)| = {dv_tol:.2e}   (écart à Ghia : {ev8:.2e})")
print(f"P : {r8['P'][-1]:.6f} (1e-8)  vs  {r10['P'][-1]:.6f} (1e-10)")

**Interprétation.** *(à rédiger après lecture des résultats)*

### 2.g Récapitulatif M1 (sauvegardé dans `results/A_M1_summary.csv`)

In [ ]:
summary = np.array([[n, runs[n]["dt"], runs[n]["n_steps"], runs[n]["t_final"], runs[n]["wall_time"],
                     runs[n]["cg_iterations"].mean(), runs[n]["max_dudt"], *ghia_errors(runs[n]),
                     runs[n]["K"][-1], runs[n]["P"][-1]] for n in GRIDS])
np.savetxt(RES / "A_M1_summary.csv", summary, delimiter=",", fmt="%.6e",
           header="N,dt,n_steps,t_final,wall_time_s,cg_iter_mean,max_dudt_final,err_ghia_u,err_ghia_v,K_final,P_final")
for row in summary:
    print(f"N = {int(row[0]):3d}   calcul = {row[4]:7.1f} s   écart Ghia (u, v) = ({row[7]:.4f}, {row[8]:.4f})"
          f"   K = {row[9]:.5f}   P = {row[10]:.5f}")

**Interprétation.** *(à rédiger après lecture des résultats)*